# Train the `lstm_modelr` ensemble member on Colab GPU

Trains the exact member the local chain would have trained (window 1318–1597,
valid 1599–1698, seed 42, tuned refit lr 1e-3) using the repo's own
`train_from_memmap.py` — identical code path, so the returned prediction
stream is drop-in compatible with the local blend.

## Before running
1. **Runtime → Change runtime type → GPU** (T4 is fine, A100 faster).
2. Upload to your Drive root (`MyDrive`):
   - `js_colab_bundle.zip` (repo code, ~small)
   - `js_pool_1318_1698.zip` (standardized data slice, ~1.2 GB)
3. Run all cells. Total: ~10 min setup + training (T4: ~1.5–2 h at
   `--batch-size 1` for exact fidelity; see the note in §4 for the 4× faster
   `--batch-size 8` variant).

## What to bring back
`MyDrive/js_regen_lstm_out/` — hand back at least
`preds/lstm_modelr_online.npz` (plus `lstm_modelr.json` for the scores).
Locally it goes to `artifacts/bench/regen_ensemble/preds/`.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Unpack code + data to LOCAL disk (never train off the FUSE mount)
!unzip -q -o /content/drive/MyDrive/js_colab_bundle.zip -d /content/project
!unzip -q -o /content/drive/MyDrive/js_pool_1318_1698.zip -d /content/
!mv -f /content/js_pool_1318_1698 /content/js_pool 2>/dev/null || true
!ls /content/project && ls /content/js_pool | head -3

In [ ]:
# 3. Deps (torch is preinstalled with CUDA on Colab)
%pip install --quiet 'numpy>=2.0' 'polars>=1.20' 'pyarrow>=15' \
  'scikit-learn>=1.5' 'tqdm>=4.66' 'xgboost>=2.1'
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 4. Train

Configured for GPU: `--batch-size 8` (keeps the GPU fed), `--epochs 30
--patience 5` (compensates for the 8x fewer optimizer steps per epoch;
early stopping picks the real stopping point). Training lr stays at the
family default 1e-3; the online-refit lr (the one with documented cliffs)
is untouched by batch size. Expect ~30-45 min on a T4.

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/project/src'
!cd /content/project && python scripts/train_from_memmap.py \
    --data /content/js_pool \
    --model lstm_modelr --resample-mode window \
    --train-lo 1318 --train-hi 1597 --valid-lo 1599 --valid-hi 1698 \
    --device cuda --seed 42 --tag lstm_modelr \
    --batch-size 8 --epochs 30 --patience 5 \
    --out /content/out

In [ ]:
# 5. Scores + copy everything back to Drive
import json, shutil
print(json.dumps(json.load(open('/content/out/lstm_modelr.json')), indent=2))
shutil.copytree('/content/out', '/content/drive/MyDrive/js_regen_lstm_out',
                dirs_exist_ok=True)
print('copied to MyDrive/js_regen_lstm_out')

## Done

Download `MyDrive/js_regen_lstm_out/preds/lstm_modelr_online.npz` (and the
`.json`) and drop them into `artifacts/bench/regen_ensemble/preds/` in the
repo. The blend script asserts row alignment via the embedded y/w, so a
misplaced file fails loudly rather than silently.